# Online Forum Multi-Label Communicative Type Classification

We classify casual online forum comments written in Bahasa Melayu, English, and Manglish from Lowyat Kopitiam based on what the user is trying to communicate.


# The 6 Categories

- **Inquiry**: Questions, seeking help, recommendations, or troubleshooting.
- **Complaint**: Venting frustration, complaining about bad service, high prices, or problems.
- **Opinion**: Personal viewpoints, beliefs, reviews, or arguments.
- **Information**: Objective facts, news, official updates, guides, or links.
- **Expressive**: Jokes, laughing, memes, greetings, or casual banter.
- **Spam**: Unwanted promotional links, referral links, or automated bot posts.


Because a single forum comment can serve multiple purposes, we use **Multi-Label Classification**.


The raw Lowyat forum records are from Hugging Face

https://huggingface.co/datasets/malaysia-ai/crawl-lowyat/resolve/main/kopitiam.json

Then, we use generative AI to help us to classify the records, and save the results in `dataset.csv`

# 1.0 Pre-processing: High-Speed Regex Masking & Formatting

This stage is to xxx

## 1.1 Initial Dataset Inspection

In [ ]:
import pandas as pd

data = pd.read_csv("dataset.csv", encoding="utf-8")

print("Total rows in the dataset: ", data.shape[0])
print(data.head(5))
print(data.sample(5))

## 1.2 Quote Block & Signature Removal

In [ ]:
import re

QUOTE_REGEX = re.compile(
    r"QUOTE\s*\([\s\S]*?\)|\[QUOTE[\s\S]*?\[/QUOTE\]", re.IGNORECASE
)
EDIT_SIG_REGEX = re.compile(r"This post has been edited by.*?(?=\n|$)", re.IGNORECASE)


def remove_quotes(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return QUOTE_REGEX.sub("", text)


def remove_lowyat_edit_signatures(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return EDIT_SIG_REGEX.sub("", text)


original_texts = data["text"].copy()
data["text"] = data["text"].apply(lambda x: remove_quotes(x))
print(
    f"Number of rows affected by quote removal: {(original_texts != data["text"]).sum()}"
)

original_texts = data["text"].copy()
data["text"] = data["text"].apply(lambda x: remove_lowyat_edit_signatures(x))
print(
    "Total rows affected by lowyat edit signature removal: ",
    (original_texts != data["text"]).sum(),
)

## 1.3 Special Element Masking

Replace specific entities with generic placeholder tags so sentence structure stays intact without leaving raw noise
- URLs & Links: `https://...` $\rightarrow$ `<URL>`
- Phone Numbers: `012-3456789, +6017...` $\rightarrow$ `<PHONE>`
- Prices / Currency: `RM150, $20` $\rightarrow$ `<PRICE>`
- Timestamps: `02:30 PM, 14:00` $\rightarrow$ `<TIME>`

### 1.3.1 Mask URL & Links

In [ ]:
URL_REGEX = re.compile(
    r"(https?://[^\s]+|www\.[^\s]+|[a-zA-Z0-9-]+\.[a-zA-Z]{2,24}[^\s]*)",
    re.IGNORECASE,
)


def mask_urls(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return URL_REGEX.sub(" <URL> ", text)


original_texts = data["text"].copy()
data["text"] = data["text"].apply(lambda x: mask_urls(x))

print("Total rows affected by masking URLs: ", (original_texts != data["text"]).sum())

### 1.3.2 Mask Phone Numbers

In [ ]:
import phonenumbers


def mask_phone_numbers(text, default_region="MY"):
    if not isinstance(text, str) or not text.strip():
        return ""

    for match in phonenumbers.PhoneNumberMatcher(text, default_region):
        phone_str = text[match.start : match.end]
        text = text.replace(phone_str, " <PHONE> ")

    return text


original_texts = data["text"].copy()
data["text"] = data["text"].apply(lambda x: mask_phone_numbers(x))

print(
    "Total rows affected by masking phone numbers: ",
    (original_texts != data["text"]).sum(),
)

### 1.3.3 Mask Prices

In [ ]:
from price_parser.parser import CURRENCY_SYMBOLS

all_symbols = set(CURRENCY_SYMBOLS)

# Add localized currency representations commonly used in regional forums
all_symbols.update(["RM", "rm", "MYR", "myr", "S$", "A$", "HK$", "NT$"])

print("All currency symbols to be masked: ", all_symbols)

In [ ]:
from price_parser import Price
from price_parser.parser import CURRENCY_SYMBOLS

all_symbols = set(CURRENCY_SYMBOLS)

# Add localized currency representations commonly used in regional forums
all_symbols.update(["RM", "rm", "MYR", "myr", "S$", "A$", "HK$", "NT$"])

# Safely escape special regex characters (e.g. $ -> \$) and sort by length descending
escaped_symbols = [re.escape(s) for s in sorted(all_symbols, key=len, reverse=True)]
DYNAMIC_CURRENCY_PATTERN = "|".join(escaped_symbols)

PRICE_REGEX = re.compile(
    rf"(?:\b\d+(?:\.\d+)?\s*(?:{DYNAMIC_CURRENCY_PATTERN})\b|(?:{DYNAMIC_CURRENCY_PATTERN})\s*\d+(?:\.\d+)?\b)",
    re.IGNORECASE,
)


def mask_prices(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return PRICE_REGEX.sub(" <PRICE> ", text)


original_texts = data["text"].copy()
data["text"] = data["text"].apply(lambda x: mask_prices(x))

print(
    "Total rows affected by masking prices: ",
    (original_texts != data["text"]).sum(),
)

### 1.3.4 Mask Timestamps

In [ ]:
TIME_REGEX = re.compile(
    r"\b(?:1[0-2]|0?[1-9]):[0-5][0-9]\s?(?:[AaPp][Mm])?\b|\b(?:[01]?[0-9]|2[0-3]):[0-5][0-9]\b"
)


def mask_times(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return TIME_REGEX.sub(" <TIME> ", text)


original_texts = data["text"].copy()
data["text"] = data["text"].apply(lambda x: mask_times(x))

print(
    "Total rows affected by masking timestamps: ",
    (original_texts != data["text"]).sum(),
)

## 1.4 Emoji Conversion

In [ ]:
import emoji


def convert_emojis(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return emoji.demojize(text)


original_texts = data["text"].copy()
data["text"] = data["text"].apply(convert_emojis)

print(
    "Total rows affected by emoji conversion: ", (original_texts != data["text"]).sum()
)

## 1.5 Lowercasing

In [ ]:
def to_lower(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return text.lower()


data["text"] = data["text"].apply(to_lower)

## 1.6 Filter Non-Latin Characters

In [ ]:
NON_LATIN_REGEX = re.compile(r"[^\x00-\x7F]+")


def strip_non_latin(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return NON_LATIN_REGEX.sub("", text)


original_texts = data["text"].copy()
data["text"] = data["text"].apply(strip_non_latin)

print(
    "Total rows affected by stripping non-Latin: ",
    (original_texts != data["text"]).sum(),
)

## 1.7 Elongated Character & Symbol Reduction

In [ ]:
# Reduces 3+ repeating letters/characters down to 2
REPEAT_CHAR_REGEX = re.compile(r"(.)\1{2,}", re.IGNORECASE)

# Reduces 2+ repeating punctuation marks down to 1
REPEAT_PUNCT_REGEX = re.compile(r"([!?.])\1+", re.IGNORECASE)

def reduce_elongated(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    text = REPEAT_CHAR_REGEX.sub(r"\1\1", text)
    text = REPEAT_PUNCT_REGEX.sub(r"\1", text)

    return 

original_texts = data["text"].copy()
data["text"] = data["text"].apply(reduce_elongated)

print(
    "Total rows affected by elongation reduction: ",
    (original_texts != data["text"]).sum(),
)

# 2.0 Pre-processing: Tokenization & Slang Normalization

## 2.1 Sentence Tokenization

In [ ]:
import nltk

# Ensure required NLTK tokenization models are downloaded
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    print("Downloading NLTK punkt tokenizer...")
    nltk.download("punkt", quiet=True)

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    print("Downloading NLTK punkt_tab tokenizer...")
    nltk.download("punkt_tab", quiet=True)

In [ ]:
from nltk.tokenize import sent_tokenize


def tokenize_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []

    return sent_tokenize(text)


original_texts = data["text"].copy()
tokenized_sentences = data["text"].apply(tokenize_sentences)

total_posts = len(tokenized_sentences)
total_sentences = tokenized_sentences.apply(len).sum()

print("Total sentences extracted across dataset: ", total_sentences)

## 2.2 Pre-built Internet Slang Normalization

We get the english slang dictionary from https://github.com/cbaziotis/ekphrasis/blob/master/ekphrasis/dicts/noslang/slangdict.py, then save it as `slangdict.json`

In [ ]:
import json

english_slangdict = json.load(open("slangdict.json", "r", encoding="utf-8"))

print("Total English Slang entries:", len(english_slangdict))
print("Examples of English Slang entries:")
for k, v in list(english_slangdict.items())[:5]:
    print("  ", k, "->", v)

In [ ]:
SLANG_MAP = {k.lower(): str(v) for k, v in english_slangdict.items() if k}

# 2. Separate multi-word keys (e.g. "what the heck") from single-word keys
multi_word_keys = [k for k in SLANG_MAP.keys() if " " in k]
multi_word_keys.sort(key=len, reverse=True)

# Tiny regex ONLY for multi-word phrases (super fast because it's small)
MULTI_WORD_REGEX = None
if multi_word_keys:
    MULTI_WORD_REGEX = re.compile(
        r"\b(" + "|".join(re.escape(k) for k in multi_word_keys) + r")\b",
        flags=re.IGNORECASE,
    )

# Regex to safely separate outside punctuation from the core word
# e.g., "a/b," -> prefix:"", core:"a/b", suffix:","
EDGE_PUNCT_REGEX = re.compile(r"^([^\w]*)(.*?)([^\w]*)$")


def normalize_slang_bulletproof(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    # Pass 1: Handle multi-word slang instantly
    if MULTI_WORD_REGEX:
        text = MULTI_WORD_REGEX.sub(
            lambda m: SLANG_MAP.get(m.group(0).lower(), m.group(0)), text
        )

    # Pass 2: Handle single-word slang with symbols
    # Split strictly by whitespace (keeps internal symbols like '/' or '\' intact)
    tokens = text.split()
    normalized_tokens = []

    for token in tokens:
        lower_token = token.lower()

        # Check if exact token is in dictionary first
        if lower_token in SLANG_MAP:
            normalized_tokens.append(SLANG_MAP[lower_token])
            continue

        # Extract prefix, core word, and suffix punctuation
        match = EDGE_PUNCT_REGEX.match(token)
        if match:
            prefix, core, suffix = match.groups()
            lower_core = core.lower()

            if lower_core in SLANG_MAP:
                # Reassemble the word with its original outside punctuation
                normalized_tokens.append(f"{prefix}{SLANG_MAP[lower_core]}{suffix}")
                continue

        # If no match found, keep original token
        normalized_tokens.append(token)

    return " ".join(normalized_tokens)


normalized_english_sentences = tokenized_sentences.apply(
    lambda sentence_list: [
        normalize_slang_bulletproof(sent) for sent in sentence_list
    ]
)

sentences_affected = sum(
    orig_sent != norm_sent
    for orig_list, norm_list in zip(
        tokenized_sentences, normalized_english_sentences
    )
    for orig_sent, norm_sent in zip(orig_list, norm_list)
)

print(
    "Total sentences affected by English slang normalization: ",
    sentences_affected,
)

## 2.3 Malay Probabilistic Normalization

We get the malay slang dictionary from https://data.mendeley.com/datasets/mgv2n2vcb9/3/files/a7b86a2f-1175-4ff0-b813-d95218534cd4, which we save as `malayslangdict.json`.

Besides, we also create our own malay slang dictionary that are not in the `malayslangdict.json` file. We save it as `custom_malay_slang.json`.

In [ ]:
malay_slang_1 = json.load(open("malayslangdict.json", "r", encoding="utf-8"))
malay_slang_2 = json.load(open("custom_malay_slang.json", "r", encoding="utf-8"))

malay_slangdict = {**malay_slang_1, **malay_slang_2}

print("Total Malay Slang entries:", len(malay_slangdict))
print("Examples of Malay Slang entries:")
for k, v in list(malay_slangdict.items())[:5]:
    print("  ", k, "->", v)

In [ ]:
SLANG_MAP = {
    str(k).lower(): str(v)
    for k, v in malay_slangdict.items()
    if k and str(k).lower() != str(v).lower()
}

# Extract multi-word terms (e.g., "tak tahu", "apa benda", "dia orang")
multi_word_keys = [k for k in SLANG_MAP.keys() if " " in k]
multi_word_keys.sort(key=len, reverse=True)

# Small regex ONLY for multi-word phrases
MULTI_WORD_REGEX = None
if multi_word_keys:
    MULTI_WORD_REGEX = re.compile(
        r"\b(" + "|".join(re.escape(k) for k in multi_word_keys) + r")\b",
        flags=re.IGNORECASE,
    )

# Edge punctuation regex to peel off outside symbols like ',' or '?'
EDGE_PUNCT_REGEX = re.compile(r"^([^\w]*)(.*?)([^\w]*)$")


def normalize_malay_slang_bulletproof(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    # Pass 1: Handle multi-word slang (e.g. "xtau" -> "tak tahu" or "apa benda" -> "amende")
    if MULTI_WORD_REGEX:
        text = MULTI_WORD_REGEX.sub(
            lambda m: SLANG_MAP.get(m.group(0).lower(), m.group(0)), text
        )

    # Pass 2: Tokenize by whitespace (preserves inner symbols like '/', '\', '-')
    tokens = text.split()
    normalized_tokens = []

    for token in tokens:
        lower_token = token.lower()

        # Direct exact match check
        if lower_token in SLANG_MAP:
            normalized_tokens.append(SLANG_MAP[lower_token])
            continue

        # Strip outside punctuation and check core word
        match = EDGE_PUNCT_REGEX.match(token)
        if match:
            prefix, core, suffix = match.groups()
            lower_core = core.lower()

            if lower_core in SLANG_MAP:
                normalized_tokens.append(f"{prefix}{SLANG_MAP[lower_core]}{suffix}")
                continue

        normalized_tokens.append(token)

    return " ".join(normalized_tokens)


normalized_malay_sentences = normalized_english_sentences.apply(
    lambda sentence_list: [
        normalize_malay_slang_bulletproof(sent) for sent in sentence_list
    ]
)

sentences_affected = sum(
    orig_sent != norm_sent
    for orig_list, norm_list in zip(
        normalized_english_sentences, normalized_malay_sentences
    )
    for orig_sent, norm_sent in zip(orig_list, norm_list)
)

print(
    "Total sentences affected by Malay slang normalization: ",
    sentences_affected,
)

## 2.4 Word Tokenization

In [ ]:
from nltk.tokenize import word_tokenize


def tokenize_words(sentence_text):
    if not isinstance(sentence_text, str) or not sentence_text.strip():
        return []
    return word_tokenize(sentence_text)


word_tokenized_posts = normalized_malay_sentences.apply(
    lambda sentence_list: [tokenize_words(sent) for sent in sentence_list]
)

total_words = sum(
    len(word) for post in word_tokenized_posts for sentence in post for word in sentence
)

print("Total word tokens generated across dataset: ", total_words)
print("\nSample Word Tokenized Output:")
for i, post in enumerate(word_tokenized_posts.iloc[:5]):
    print(" ", i, ". ", post)